In [32]:
import pandas as pd
import numpy as np
from prophet import Prophet
import matplotlib.pyplot as plt

# -------------------------
# CONFIG
# -------------------------
CSV_PATH = "HousePrice.csv"     # change this
HORIZON_WEEKS = 104            # 2 years ahead

TARGET_METROS = [
    "Atlanta, GA metro area",
    "Chicago, IL metro area",
    "Phoenix, AZ metro area",
    "Dallas, TX metro area",
    "Houston, TX metro area",
    "Minneapolis, MN metro area",
]

In [33]:
# -------------------------
# ERROR METRICS
# -------------------------
def compute_error_metrics(y_true, y_pred):
    """Return MAE, RMSE, MAPE (ignores zero / NaN true values for MAPE)."""
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=float)

    mask = ~np.isnan(y_true) & ~np.isnan(y_pred)
    y_true = y_true[mask]
    y_pred = y_pred[mask]

    if len(y_true) == 0:
        return np.nan, np.nan, np.nan

    mae = np.mean(np.abs(y_true - y_pred))
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))

    nonzero = y_true != 0
    if np.any(nonzero):
        mape = np.mean(np.abs((y_true[nonzero] - y_pred[nonzero]) / y_true[nonzero])) * 100
    else:
        mape = np.nan

    return mae, rmse, mape

In [34]:
# -------------------------
# LOAD + FILTER
# -------------------------
df = pd.read_csv(CSV_PATH)
df["PERIOD_END"] = pd.to_datetime(df["PERIOD_END"])

df_metros = df[df["REGION_NAME"].isin(TARGET_METROS)].copy()

if df_metros.empty:
    raise ValueError("No rows found for the specified metros. Check REGION_NAME values.")


In [35]:
# -------------------------
# HYPERPARAMETER GRID
# -------------------------
param_grid = [
    # low flexibility, stronger regularization
    {"changepoint_prior_scale": 0.05, "seasonality_prior_scale": 5.0, "yearly_seasonality": True},
    {"changepoint_prior_scale": 0.05, "seasonality_prior_scale": 10.0, "yearly_seasonality": True},

    # moderate flexibility (similar to default)
    {"changepoint_prior_scale": 0.1, "seasonality_prior_scale": 10.0, "yearly_seasonality": True},
    {"changepoint_prior_scale": 0.1, "seasonality_prior_scale": 15.0, "yearly_seasonality": True},

    # more flexible trend (can track sharper shifts)
    {"changepoint_prior_scale": 0.2, "seasonality_prior_scale": 10.0, "yearly_seasonality": True},
    {"changepoint_prior_scale": 0.2, "seasonality_prior_scale": 15.0, "yearly_seasonality": True},
]


In [36]:
# -------------------------
# MAIN LOOP – PER METRO
# -------------------------
all_forecasts = []
validation_results = []

for metro in sorted(df_metros["REGION_NAME"].unique()):
    print(f"\n==============================")
    print(f"Processing metro: {metro}")
    print(f"==============================")

    sub = df_metros[df_metros["REGION_NAME"] == metro].copy()
    sub = sub.sort_values("PERIOD_END")

    ts = sub[["PERIOD_END", "MEDIAN_SALE_PRICE"]].rename(
        columns={"PERIOD_END": "ds", "MEDIAN_SALE_PRICE": "y"}
    )
    ts["y"] = ts["y"].replace(0, np.nan)
    ts = ts.dropna()

    n = len(ts)
    if n < 40:
        print(f"Skipping {metro}: only {n} valid points.")
        continue

    # ---- Train/Validation Split ----
    test_size = 52 if n > 80 else max(10, n // 4)
    ts_train = ts.iloc[:-test_size].copy()
    ts_test = ts.iloc[-test_size:].copy()

    best_mape = np.inf
    best_params = None
    best_metrics = None

    # -------------------------
    # GRID SEARCH OVER PARAMS
    # -------------------------
    for params in param_grid:
        try:
            m_val = Prophet(
                seasonality_mode="additive",
                yearly_seasonality=params["yearly_seasonality"],
                weekly_seasonality=False,
                daily_seasonality=False,
                changepoint_prior_scale=params["changepoint_prior_scale"],
                seasonality_prior_scale=params["seasonality_prior_scale"],
            )
            m_val.fit(ts_train)

            future_val = m_val.make_future_dataframe(periods=test_size, freq="W")
            fc_val = m_val.predict(future_val)

            fc_val_subset = fc_val[fc_val["ds"].isin(ts_test["ds"])]
            fc_val_subset = fc_val_subset.sort_values("ds")
            ts_test_sorted = ts_test.sort_values("ds")

            min_len = min(len(ts_test_sorted), len(fc_val_subset))
            if min_len == 0:
                continue

            y_true = ts_test_sorted["y"].iloc[:min_len].values
            y_pred = fc_val_subset["yhat"].iloc[:min_len].values

            mae, rmse, mape = compute_error_metrics(y_true, y_pred)

            if not np.isnan(mape) and mape < best_mape:
                best_mape = mape
                best_params = params
                best_metrics = (mae, rmse, mape)
        except Exception as e:
            print(f"Param set {params} failed for {metro}: {e}")
            continue

    if best_params is None:
        print(f"No valid params found for {metro}, skipping.")
        continue

    mae, rmse, mape = best_metrics
    print(f"Best params for {metro}: {best_params}")
    print(f"Validation → MAE: {mae:,.2f}, RMSE: {rmse:,.2f}, MAPE: {mape:,.2f}%")

    validation_results.append({
        "REGION_NAME": metro,
        "n_points": n,
        "test_size": test_size,
        "MAE": mae,
        "RMSE": rmse,
        "MAPE_percent": mape,
        "changepoint_prior_scale": best_params["changepoint_prior_scale"],
        "seasonality_prior_scale": best_params["seasonality_prior_scale"],
        "yearly_seasonality": best_params["yearly_seasonality"],
    })

    # -------------------------
    # FINAL MODEL ON FULL DATA WITH BEST PARAMS
    # -------------------------
    m_full = Prophet(
        seasonality_mode="additive",
        yearly_seasonality=best_params["yearly_seasonality"],
        weekly_seasonality=False,
        daily_seasonality=False,
        changepoint_prior_scale=best_params["changepoint_prior_scale"],
        seasonality_prior_scale=best_params["seasonality_prior_scale"],
    )
    m_full.fit(ts)

    future_full = m_full.make_future_dataframe(periods=HORIZON_WEEKS, freq="W")
    fc_full = m_full.predict(future_full)

    future_forecast = fc_full.tail(HORIZON_WEEKS).copy()
    future_forecast["REGION_NAME"] = metro

    future_forecast = future_forecast[
        ["REGION_NAME", "ds", "yhat", "yhat_lower", "yhat_upper"]
    ].rename(columns={"ds": "date"})

    # Round to integers
    future_forecast["yhat"] = future_forecast["yhat"].round().astype(int)
    future_forecast["yhat_lower"] = future_forecast["yhat_lower"].round().astype(int)
    future_forecast["yhat_upper"] = future_forecast["yhat_upper"].round().astype(int)

    all_forecasts.append(future_forecast)

20:38:04 - cmdstanpy - INFO - Chain [1] start processing
20:38:04 - cmdstanpy - INFO - Chain [1] done processing
20:38:04 - cmdstanpy - INFO - Chain [1] start processing



Processing metro: Atlanta, GA metro area


20:38:04 - cmdstanpy - INFO - Chain [1] done processing
20:38:04 - cmdstanpy - INFO - Chain [1] start processing
20:38:04 - cmdstanpy - INFO - Chain [1] done processing
20:38:04 - cmdstanpy - INFO - Chain [1] start processing
20:38:04 - cmdstanpy - INFO - Chain [1] done processing
20:38:04 - cmdstanpy - INFO - Chain [1] start processing
20:38:04 - cmdstanpy - INFO - Chain [1] done processing
20:38:04 - cmdstanpy - INFO - Chain [1] start processing
20:38:04 - cmdstanpy - INFO - Chain [1] done processing
20:38:05 - cmdstanpy - INFO - Chain [1] start processing
20:38:05 - cmdstanpy - INFO - Chain [1] done processing


Best params for Atlanta, GA metro area: {'changepoint_prior_scale': 0.2, 'seasonality_prior_scale': 15.0, 'yearly_seasonality': True}
Validation → MAE: 14,498.48, RMSE: 16,085.88, MAPE: 3.67%


20:38:05 - cmdstanpy - INFO - Chain [1] start processing
20:38:05 - cmdstanpy - INFO - Chain [1] done processing
20:38:05 - cmdstanpy - INFO - Chain [1] start processing


...

Best params for Houston, TX metro area: {'changepoint_prior_scale': 0.05, 'seasonality_prior_scale': 5.0, 'yearly_seasonality': True}
Validation → MAE: 6,199.84, RMSE: 7,475.27, MAPE: 1.83%

Processing metro: Minneapolis, MN metro area


20:38:08 - cmdstanpy - INFO - Chain [1] start processing
20:38:08 - cmdstanpy - INFO - Chain [1] done processing
20:38:08 - cmdstanpy - INFO - Chain [1] start processing
20:38:08 - cmdstanpy - INFO - Chain [1] done processing
20:38:08 - cmdstanpy - INFO - Chain [1] start processing
20:38:08 - cmdstanpy - INFO - Chain [1] done processing
20:38:09 - cmdstanpy - INFO - Chain [1] start processing
20:38:09 - cmdstanpy - INFO - Chain [1] done processing
20:38:09 - cmdstanpy - INFO - Chain [1] start processing
20:38:09 - cmdstanpy - INFO - Chain [1] done processing
20:38:09 - cmdstanpy - INFO - Chain [1] start processing
20:38:09 - cmdstanpy - INFO - Chain [1] done processing
20:38:09 - cmdstanpy - INFO - Chain [1] start processing


Best params for Minneapolis, MN metro area: {'changepoint_prior_scale': 0.05, 'seasonality_prior_scale': 5.0, 'yearly_seasonality': True}
Validation → MAE: 3,161.42, RMSE: 3,892.45, MAPE: 0.82%

Processing metro: Phoenix, AZ metro area


20:38:09 - cmdstanpy - INFO - Chain [1] done processing
20:38:09 - cmdstanpy - INFO - Chain [1] start processing
20:38:09 - cmdstanpy - INFO - Chain [1] done processing
20:38:09 - cmdstanpy - INFO - Chain [1] start processing
20:38:09 - cmdstanpy - INFO - Chain [1] done processing
20:38:09 - cmdstanpy - INFO - Chain [1] start processing
20:38:10 - cmdstanpy - INFO - Chain [1] done processing
20:38:10 - cmdstanpy - INFO - Chain [1] start processing
20:38:10 - cmdstanpy - INFO - Chain [1] done processing
20:38:10 - cmdstanpy - INFO - Chain [1] start processing
20:38:10 - cmdstanpy - INFO - Chain [1] done processing
20:38:10 - cmdstanpy - INFO - Chain [1] start processing
20:38:10 - cmdstanpy - INFO - Chain [1] done processing


Best params for Phoenix, AZ metro area: {'changepoint_prior_scale': 0.05, 'seasonality_prior_scale': 5.0, 'yearly_seasonality': True}
Validation → MAE: 12,890.68, RMSE: 16,089.57, MAPE: 2.81%


In [37]:
# -------------------------
# STACK WEEKLY FORECASTS
# -------------------------
if not all_forecasts:
    raise ValueError("No forecasts generated. Check data and metro names.")

weekly_forecasts_df = pd.concat(all_forecasts, ignore_index=True)

print("\nSample weekly forecasts (rounded):")
print(weekly_forecasts_df.head())


Sample weekly forecasts (rounded):
              REGION_NAME       date    yhat  yhat_lower  yhat_upper
0  Atlanta, GA metro area 2025-09-28  393776      389668      397924
1  Atlanta, GA metro area 2025-10-05  393770      389571      397898
2  Atlanta, GA metro area 2025-10-12  393588      389389      397938
3  Atlanta, GA metro area 2025-10-19  393084      388874      397226
4  Atlanta, GA metro area 2025-10-26  392378      388065      396712


In [38]:
# -------------------------
# YEARLY AGGREGATES (ROUNDED)
# -------------------------
weekly_forecasts_df["year"] = weekly_forecasts_df["date"].dt.year

yearly_forecasts_df = (
    weekly_forecasts_df
    .groupby(["REGION_NAME", "year"], as_index=False)["yhat"]
    .median()
    .rename(columns={"yhat": "median_sale_price_forecast"})
)

yearly_forecasts_df["median_sale_price_forecast"] = (
    yearly_forecasts_df["median_sale_price_forecast"].round().astype(int)
)

print("\nSample yearly forecasts (rounded):")
print(yearly_forecasts_df.head())


Sample yearly forecasts (rounded):
              REGION_NAME  year  median_sale_price_forecast
0  Atlanta, GA metro area  2025                      390922
1  Atlanta, GA metro area  2026                      393716
2  Atlanta, GA metro area  2027                      396656
3  Chicago, IL metro area  2025                      362282
4  Chicago, IL metro area  2026                      386012


In [39]:
# -------------------------
# SAVE RESULTS
# -------------------------
pd.DataFrame(validation_results).to_csv(
    "prophet_tuned_validation_metrics_6_metros.csv", index=False
)
weekly_forecasts_df.to_csv(
    "weekly_median_sale_price_prophet_tuned_6_metros_rounded.csv", index=False
)
yearly_forecasts_df.to_csv(
    "yearly_median_sale_price_prophet_tuned_6_metros_rounded.csv", index=False
)